<!-- ============================================================ -->
<!-- NOTEBOOK HEADER — MLOps Introductory Course on GCP           -->
<!-- ============================================================ -->

<div style="border-bottom: 3px solid #4285F4; padding-bottom: 12px; margin-bottom: 20px;">

<div style="display: flex; align-items: center; justify-content: space-between;">
  <div>
    <img src="https://www.isae-supaero.fr/wp-content/uploads/2025/03/logo.svg" width="180">
  </div>
  <div style="text-align: right;">
    <img src="https://user-images.githubusercontent.com/63151412/167391313-4683cc69-2bf6-4597-b767-5c18e2bbbfa0.png" width="180">
  </div>
</div>

# Lab 03b — Online Endpoints & Load Testing

**Course:** MLOps Introductory Course on GCP · M2 Data Science · ISAE-SUPAERO  
**Lab created by:** Headmind Partners AI & Blockchain  
**Estimated duration:** ~1h15

</div>

## 📋 Lab Overview

In Lab 03a you trained a model and ran batch predictions — a pattern suited for offline workloads. But many applications require **real-time predictions** with low latency: a mobile app classifying a photo, a fraud detection system scoring transactions, or a recommendation engine serving results on page load. This lab focuses on the **online serving** side of MLOps.

Beyond basic deployment (which you practiced in Lab 02), production endpoints require **traffic management** (A/B testing, canary releases), **autoscaling**, and **load testing** to ensure they meet SLAs under heavy traffic.

### Learning Objectives

1. Deploy a model to a Vertex AI endpoint with **traffic splitting** and **autoscaling** configuration.
2. Send **online predictions** via the SDK and evaluate results.
3. Understand how to wrap a Vertex AI endpoint with **FastAPI** to build a custom API layer.
4. Run **load tests** with Locust and interpret throughput / latency metrics.
5. Reason about deployment strategies: canary, blue-green, and A/B testing.

### Notebook Structure

| # | Section | Focus |
|---|---------|-------|
| 0 | Setup | Install dependencies, imports, retrieve model from Lab 05 |
| 1 | Deploy to Endpoint | Create endpoint, deploy with traffic split & autoscaling |
| 2 | Online Predictions | Send requests via SDK, evaluate results |
| 3 | FastAPI Wrapper | Analyze the API layer, run locally, send requests |
| 4 | Load Testing with Locust | Run Locust, interpret results, discuss scaling |
| 5 | Cleanup | Undeploy model, delete endpoint |

### How to Read This Notebook

- **`# TODO`** — Code you need to write. Look for the `######` delimiters.
- **`✏️ Question`** — A conceptual question. Write your answer in the markdown cell below it.
- Cells **without** a TODO are provided — read them, run them, and make sure you understand them.
- Documentation links are provided in 📖 callouts whenever a new API is introduced.

---
## 0 · Setup

### 0.1 Install dependencies

In [69]:
%pip install --upgrade --quiet google-cloud-aiplatform google-cloud-storage pillow numpy fastapi uvicorn locust

### 0.2 Imports

In [70]:
import os
import json
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
from PIL import Image

from google.cloud import aiplatform

CLASS_NAMES = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck",
]

print(f"Vertex AI SDK version: {aiplatform.__version__}")

Vertex AI SDK version: 1.140.0


### 0.3 Configuration & model retrieval

Use the same GCP project settings as Lab 05. Then retrieve the model you trained there.

In [71]:
# ── Constants ──
PROJECT_ID = "isae-sdd-481407"           # @param {type:"string"}
LOCATION = "europe-west3"                # @param {type:"string"}
BUCKET_URI = "gs://bucket-lab03"    # @param {type:"string"}

##############################  TODO  ##############################
# Set your_name to a unique lowercase identifier (e.g. first letter of first name + last name).
your_name ="mregaieg"  # TODO
####################################################################

aiplatform.init(
    project=PROJECT_ID,
    location=LOCATION,
    staging_bucket=BUCKET_URI,
)
print(f"✅ Vertex AI initialized — project={PROJECT_ID}, location={LOCATION}")

✅ Vertex AI initialized — project=isae-sdd-481407, location=europe-west3


In [72]:
##############################  TODO  ##############################
# Retrieve the model trained in Lab 03a from the Model Registry.
# Option A: Use the resource_name you noted at the end of Lab 03a.
# Option B: List models and filter by display name.
#
# If using Option A:
#   model = aiplatform.Model("projects/.../models/...")
#
# If using Option B:
#   models = aiplatform.Model.list(filter='display_name="cifar10-model"')
#   model = models[0]

model = aiplatform.Model("projects/588027604890/locations/europe-west3/models/3825231739358281728")  # TODO

####################################################################

print(f"✅ Model loaded: {model.display_name} ({model.resource_name})")

✅ Model loaded: cifar10-model (projects/588027604890/locations/europe-west3/models/3825231739358281728)


---
## 1 · Deploy to an Online Endpoint

In Lab 02b, you created an endpoint and deployed a model with default settings. Here we go further: you will configure **traffic splitting** and **autoscaling** — two critical features for production deployments.

### 1.1 Deployment strategies

Before writing code, let's understand three common deployment strategies:

| Strategy | How it works | Use case |
|----------|-------------|----------|
| **Canary** | Route a small % of traffic to the new model, then gradually increase | Safe rollout — catch issues before they affect all users |
| **Blue-Green** | Two identical environments; switch traffic all at once | Zero-downtime releases with instant rollback |
| **A/B Testing** | Split traffic between models to compare performance on real data | Measure business impact of a new model |

Vertex AI supports these patterns through the `traffic_split` parameter when deploying models to an endpoint.

### 1.2 Create an endpoint

> 📖 **Docs:** [`Endpoint.create()`](https://cloud.google.com/python/docs/reference/aiplatform/latest/google.cloud.aiplatform.Endpoint#google_cloud_aiplatform_Endpoint_create)

In [73]:
##############################  TODO  ##############################
# Create an endpoint with a descriptive display name, e.g. "cifar10-endpoint-{your_name}"
endpoint = aiplatform.Endpoint.create(
    "cifar10-endpoint-mohamed" # TODO
)
####################################################################

print(f"✅ Endpoint created: {endpoint.resource_name}")

✅ Endpoint created: projects/588027604890/locations/europe-west3/endpoints/4227721164845219840


### 1.3 Deploy with traffic split and autoscaling

When deploying, the `traffic_split` dictionary controls how incoming requests are distributed. The key `"0"` refers to the model being deployed *in this call*; existing deployed model IDs can be used as other keys.

**Autoscaling** is controlled by `min_replica_count` and `max_replica_count`. Vertex AI automatically adjusts the number of serving replicas between these bounds based on traffic load.

> 📖 **Docs:** [`Endpoint.deploy()`](https://cloud.google.com/python/docs/reference/aiplatform/latest/google.cloud.aiplatform.Endpoint#google_cloud_aiplatform_Endpoint_deploy)

In [74]:
##############################  TODO  ##############################
# Deploy the model to the endpoint.
# Configure:
#   - deployed_model_display_name: a name for the deployed model
#   - traffic_split: send 100% traffic to this model ({"0": 100})
#   - machine_type: "n1-standard-4"
#   - min_replica_count: 1
#   - max_replica_count: 3  (allows autoscaling up to 3 replicas)

DEPLOYED_NAME = "cifar10-deployed"

response = endpoint.deploy(
    model=model,
    deployed_model_display_name=f"{DEPLOYED_NAME}-{your_name}",  # TODO
    traffic_split={"0": 100},                # TODO
    machine_type="n1-standard-4",                 # TODO
    min_replica_count=1,            # TODO
    max_replica_count=3,            # TODO
)
####################################################################

print(f"✅ Model deployed to endpoint.")

✅ Model deployed to endpoint.


> ⏳ **Deployment takes 5–10 minutes.** While you wait, answer the question below.

**✏️ Question 1 — Traffic splitting & autoscaling**

a) You deployed with `traffic_split={"0": 100}`. Suppose you train an improved model and want to do a **canary release**, sending only 10% of traffic to the new model. How would you configure `traffic_split`? (Hint: you need the deployed model ID of the existing model.)

b) You set `max_replica_count=3`. What signals does Vertex AI use to decide when to scale up? What happens when traffic drops?

---
*Your answer:*

**a)** For a canary release, the traffic split should send **90% of the requests to the existing deployed model** and **10% to the new model**. The existing model has the ID **3825231739358281728**, while `"0"` represents the new model being deployed.

**b)** With `max_replica_count=3`, Vertex AI automatically scales the number of replicas based on resource utilization. If traffic increases and the CPU or GPU usage goes above the target level, Vertex AI can add replicas up to a maximum of three. When the load decreases, it can scale down again but not below the minimum number of replicas.



---

### 1.4 Inspect the deployment

In [75]:
# Retrieve deployment details
deployed_models = endpoint.gca_resource.deployed_models
for dm in deployed_models:
    print(f"Deployed model ID:    {dm.id}")
    print(f"Display name:         {dm.display_name}")
    print(f"Model resource:       {dm.model}")
    print(f"Machine type:         {dm.dedicated_resources.machine_spec.machine_type}")
    print(f"Min replicas:         {dm.dedicated_resources.min_replica_count}")
    print(f"Max replicas:         {dm.dedicated_resources.max_replica_count}")
    print()

Deployed model ID:    7931949450543497216
Display name:         cifar10-deployed-mregaieg
Model resource:       projects/588027604890/locations/europe-west3/models/3825231739358281728
Machine type:         n1-standard-4
Min replicas:         1
Max replicas:         3



---
## 2 · Online Predictions

With the model deployed, we can send prediction requests via the SDK. Unlike batch prediction, online prediction returns results **synchronously** — the response comes back immediately.

### 2.1 Prepare test data

In [76]:
# Download test images (if not already present from Lab 03a)
if not os.path.exists("cifar_test_images"):
    ! gsutil -m cp -r gs://cloud-samples-data/ai-platform-unified/cifar_test_images .

# Load and preprocess
IMAGE_DIRECTORY = "cifar_test_images"
image_files = [f for f in os.listdir(IMAGE_DIRECTORY) if f.endswith(".jpg")]
image_data = [np.asarray(Image.open(os.path.join(IMAGE_DIRECTORY, f))) for f in image_files]
x_test = [(img / 255.0).astype(np.float32).tolist() for img in image_data]
y_test = [int(f.split("_")[1]) for f in image_files]

print(f"✅ Loaded {len(x_test)} test images")
print(f"True labels: {[CLASS_NAMES[y] for y in y_test]}")

✅ Loaded 10 test images
True labels: ['horse', 'airplane', 'bird', 'cat', 'bird', 'deer', 'bird', 'bird', 'airplane', 'dog']


### 2.2 Send predictions

> 📖 **Docs:** [`Endpoint.predict()`](https://cloud.google.com/python/docs/reference/aiplatform/latest/google.cloud.aiplatform.Endpoint#google_cloud_aiplatform_Endpoint_predict)

In [77]:
##############################  TODO  ##############################
# Send all test images to the endpoint for online prediction.
# Use endpoint.predict() with the instances parameter.

predictions = endpoint.predict(instances=x_test)  # TODO
####################################################################

print(f"✅ Received {len(predictions.predictions)} predictions")

✅ Received 10 predictions


### 2.3 Evaluate results

In [78]:
##############################  TODO  ##############################
# Evaluate the online predictions.
# 1. Use np.argmax to get predicted labels
# 2. Compare with y_test and compute accuracy

y_predicted = [np.argmax(prediction) for prediction in predictions.predictions]  # TODO
correct = np.sum(np.array(y_predicted) == np.array(y_test))      # TODO
accuracy = correct/len(predictions.predictions)     # TODO
####################################################################

print(f"Accuracy: {accuracy:.0%} ({correct}/{len(y_test)})")
for i, (true, pred) in enumerate(zip(y_test, y_predicted)):
    status = "✅" if true == pred else "❌"
    print(f"  {status} Image {i}: true={CLASS_NAMES[true]}, predicted={CLASS_NAMES[pred]}")

Accuracy: 50% (5/10)
  ✅ Image 0: true=horse, predicted=horse
  ✅ Image 1: true=airplane, predicted=airplane
  ❌ Image 2: true=bird, predicted=horse
  ❌ Image 3: true=cat, predicted=dog
  ✅ Image 4: true=bird, predicted=bird
  ❌ Image 5: true=deer, predicted=horse
  ✅ Image 6: true=bird, predicted=bird
  ❌ Image 7: true=bird, predicted=frog
  ✅ Image 8: true=airplane, predicted=airplane
  ❌ Image 9: true=dog, predicted=frog


**✏️ Question 2 — Batch vs. Online inference**

a) You just ran the same model with both batch (lab03) and online inference. Compare the two approaches in terms of latency, throughput, cost, and appropriate use cases.

b) For the e-commerce scenario described in Lab 03a (classifying product images overnight), which approach is more appropriate and why?

---
*Your answer:*

**a)** Online inference gives predictions almost instantly, so it is better when low latency is important. In this lab, the streaming prediction felt immediate, while batch prediction took more time because the job had to be created, scheduled, and processed before producing the results. Batch inference is usually better for high throughput because it can process a large number of examples at once. It can also be more cost-effective for large offline workloads, while online inference is more suitable when predictions are needed in real time.

**b)** For the e-commerce scenario in Lab 03a, batch inference is more appropriate because the product images are classified overnight. In that case, immediate predictions are not necessary, so it makes more sense to use a method designed for processing many images together rather than serving predictions one by one in real time.


---

---
## 3 · FastAPI Wrapper

In production, you rarely expose a Vertex AI endpoint directly to end users. Instead, you build an **API layer** that handles authentication, input validation, response formatting, and business logic. FastAPI is a popular Python framework for building such APIs.

### 3.1 Why wrap a Vertex AI endpoint?

There are several reasons to add an API layer in front of your ML endpoint:
- **Authentication & authorization**: control who can access predictions
- **Input validation**: reject malformed requests before they reach the model
- **Response transformation**: convert raw model output into business-friendly responses (e.g., label names instead of class indices)
- **Rate limiting & caching**: protect the endpoint from abuse
- **Decoupling**: your application code depends on *your* API contract, not Vertex AI's

### 3.2 Analyze the API

Below is the FastAPI application that wraps the Vertex AI endpoint. Read through it and answer the question that follows.

> 📖 **Docs:** [FastAPI documentation](https://fastapi.tiangolo.com/)

In [79]:
# Display the api.py file
print(open("api.py").read() if os.path.exists("api.py") else "api.py not found — please copy it to your working directory.")

"""
FastAPI wrapper for the Vertex AI CIFAR-10 endpoint.

Usage:
  On Vertex AI Workbench:  Started programmatically from the notebook.
  On local machine:        uvicorn api:app --reload --port 8081
"""

from fastapi import FastAPI, HTTPException
import numpy as np
from pydantic import BaseModel
from google.cloud import aiplatform
from typing import List

app = FastAPI()

# ── Configuration ──
PROJECT_ID = "isae-sdd-481407"
LOCATION = "europe-west3"
BUCKET_URI = "gs://bucket-lab03"
ENDPOINT_ID = "2502842507562319872"

# ── Initialize Vertex AI ──
aiplatform.init(project=PROJECT_ID, location=LOCATION, staging_bucket=BUCKET_URI)
endpoint = aiplatform.Endpoint(ENDPOINT_ID)

# ── CIFAR-10 class names ──
CLASS_NAMES = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck",
]

# ── Schemas ──
class PredictionRequest(BaseModel):
    """A single 32x32x3 image as a nested list of floats."""
    image: List[List[List[float]]]


class PredictionRespons

If `api.py` is not present, create it with the cell below.

In [80]:
%%writefile api.py
"""
FastAPI wrapper for the Vertex AI CIFAR-10 endpoint.

Usage:
  On Vertex AI Workbench:  Started programmatically from the notebook.
  On local machine:        uvicorn api:app --reload --port 8081
"""

from fastapi import FastAPI, HTTPException
import numpy as np
from pydantic import BaseModel
from google.cloud import aiplatform
from typing import List

app = FastAPI()

# ── Configuration ──
PROJECT_ID = "your-project-id"
LOCATION = "us-central1"
BUCKET_URI = "gs://your-bucket"
ENDPOINT_ID = "your-endpoint-id"

# ── Initialize Vertex AI ──
aiplatform.init(project=PROJECT_ID, location=LOCATION, staging_bucket=BUCKET_URI)
endpoint = aiplatform.Endpoint(ENDPOINT_ID)

# ── CIFAR-10 class names ──
CLASS_NAMES = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck",
]

# ── Schemas ──
class PredictionRequest(BaseModel):
    """A single 32x32x3 image as a nested list of floats."""
    image: List[List[List[float]]]


class PredictionResponse(BaseModel):
    """The predicted class index, name, and full probability vector."""
    predicted_class: int
    class_name: str
    probabilities: List[float]


@app.get("/health")
def health_check():
    return {"status": "healthy"}


@app.post("/predict", response_model=PredictionResponse)
async def predict(request: PredictionRequest):
    try:
        result = endpoint.predict(instances=[request.image])
        probs = result.predictions[0]
        idx = int(np.argmax(probs))
        return PredictionResponse(
            predicted_class=idx,
            class_name=CLASS_NAMES[idx],
            probabilities=probs,
        )
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Prediction error: {e}")

Overwriting api.py


In [81]:
# Patch api.py with your actual configuration
ENDPOINT_ID = endpoint.name  # The numeric endpoint ID

with open("api.py", "r") as f:
    content = f.read()

content = content.replace('PROJECT_ID = "your-project-id"', f'PROJECT_ID = "{PROJECT_ID}"')
content = content.replace('LOCATION = "us-central1"', f'LOCATION = "{LOCATION}"')
content = content.replace('BUCKET_URI = "gs://your-bucket"', f'BUCKET_URI = "{BUCKET_URI}"')
content = content.replace('ENDPOINT_ID = "your-endpoint-id"', f'ENDPOINT_ID = "{ENDPOINT_ID}"')

with open("api.py", "w") as f:
    f.write(content)

print(f"✅ api.py patched with ENDPOINT_ID={ENDPOINT_ID}")

✅ api.py patched with ENDPOINT_ID=4227721164845219840


### 3.3 Start the FastAPI server from the notebook

Since we are running on Vertex AI Workbench, we cannot easily open `localhost` in a browser. Instead, we start the FastAPI server **as a background process** and interact with it programmatically via `requests`.

> 💡 **Tip:** The server runs in a background thread so you can continue executing notebook cells.

In [82]:
import threading, time, uvicorn, requests, importlib
import api as api_module
importlib.reload(api_module)

def run_server():
    uvicorn.run(api_module.app, host="127.0.0.1", port=8081)

thread = threading.Thread(target=run_server, daemon=True)
thread.start()
time.sleep(5)

try:
    resp = requests.get("http://127.0.0.1:8081/health", timeout=5)
    print(f"✅ FastAPI server is running — health check: {resp.json()}")
except Exception as e:
    print(f"❌ {e}")

INFO:     Started server process [35874]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('127.0.0.1', 8081): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


INFO:     127.0.0.1:39652 - "GET /health HTTP/1.1" 200 OK
✅ FastAPI server is running — health check: {'status': 'healthy'}


In [84]:
# Test with a single image
single_image = x_test[0]
response = requests.post(
    "http://localhost:8081/predict",
    json={"image": single_image},
    timeout=30,
)

if response.status_code == 200:
    result = response.json()
    print(f"✅ Prediction: {result['class_name']} (class {result['predicted_class']})")
    print(f"   True label: {CLASS_NAMES[y_test[0]]}")
    print(f"   Top-3 probabilities: {sorted(enumerate(result['probabilities']), key=lambda x: -x[1])[:3]}")
else:
    print(f"❌ Error {response.status_code}: {response.text}")

INFO:     127.0.0.1:59666 - "POST /predict HTTP/1.1" 200 OK
✅ Prediction: horse (class 7)
   True label: horse
   Top-3 probabilities: [(7, 0.217564628), (6, 0.205005571), (4, 0.106844679)]


**✏️ Question 3 — API design**

a) The original `api.py` from the old lab returned only the raw class index (an integer). The version above returns the class name and full probability vector. Why is returning richer responses a better practice for production APIs?

b) What is the purpose of the `/health` endpoint? In which deployment scenarios is it used?

c) Name two additional features you would add to this API before deploying it to production.

---
*Your answer:*

**a)** Returning richer responses is better for a production API because it gives more useful information to the client. Instead of only returning the predicted class index, the API also returns the class name and the probability scores, which makes the result easier to understand and easier to use in an application. It also helps with debugging and with checking how confident the model is.

**b)** The `/health` endpoint is used to check whether the API is running correctly. It is useful in deployment scenarios where a system needs to verify that the service is available before sending requests to it, for example during startup, monitoring, or load balancing.

**c)** Two useful features to add before production would be authentication and logging. Authentication would protect the API from unauthorized access, and logging would help track requests, errors, and model behavior.

---

---
## 4 · Load Testing with Locust

Before going to production, you need to verify that your endpoint can handle the expected traffic. **Load testing** simulates concurrent users to measure throughput, latency, and failure rates under increasing load.

> 📖 **Docs:** [Locust](https://docs.locust.io/en/stable/)

### 4.1 How Locust works

Locust spawns virtual **users** that repeatedly execute tasks (in our case, sending prediction requests). Each user waits a random time between requests (the `wait_time`). Key metrics Locust reports:

- **RPS** (requests per second) — throughput.
- **Median latency** — the typical response time.
- **P95 latency** — 95% of requests are faster than this. Captures tail latency.
- **Failure rate** — percentage of requests that failed.

### 4.2 Headless mode

Locust normally provides a web UI at `localhost:8089`, but on Vertex AI Workbench you cannot access local ports in a browser. Instead, we use **headless mode** (`--headless`), which runs the load test from the command line and prints results to stdout. This works perfectly from a notebook cell.

> 💡 **Alternative — local VS Code setup:** If you want the full Locust web UI with interactive charts, see the appendix at the end of this section for instructions on running the notebook locally with VS Code.

### 4.3 Write the Locust test file

In [85]:
%%writefile locustfile.py
"""
Locust load test for the CIFAR-10 FastAPI wrapper.

Headless usage (from notebook):
    locust -f locustfile.py --host http://localhost:8081 --headless -u 5 -r 1 --run-time 60s --csv results

Web UI usage (from local machine):
    locust -f locustfile.py --host http://localhost:8081
"""

import os
import json
import numpy as np
from PIL import Image
from locust import HttpUser, task, between


def load_test_images():
    """Load and preprocess CIFAR-10 test images."""
    IMAGE_DIRECTORY = "cifar_test_images"
    image_files = sorted([f for f in os.listdir(IMAGE_DIRECTORY) if f.endswith(".jpg")])
    images = [np.asarray(Image.open(os.path.join(IMAGE_DIRECTORY, f))) for f in image_files]
    return [(img / 255.0).astype(np.float32).tolist() for img in images]


TEST_IMAGES = load_test_images()


class CifarUser(HttpUser):
    """Simulated user that sends prediction requests."""

    wait_time = between(1, 3)

    @task
    def predict_single_image(self):
        """Send a single random image for prediction."""
        idx = np.random.randint(len(TEST_IMAGES))
        payload = {"image": TEST_IMAGES[idx]}
        self.client.post("/predict", json=payload)

Overwriting locustfile.py


### 4.4 Run load tests

We will run three scenarios with increasing numbers of concurrent users. Each test runs for 60 seconds in headless mode. Locust writes CSV result files that we can analyze afterward.

The key Locust CLI flags:
- `--headless` — no web UI, run from the command line.
- `-u N` — number of concurrent users to simulate.
- `-r N` — spawn rate (users added per second).
- `--run-time 60s` — stop after 60 seconds.
- `--csv prefix` — write results to CSV files for analysis.

**Scenario 1 — Baseline (5 users)**

In [86]:
# Run load test: 5 concurrent users for 60 seconds
! locust -f locustfile.py \
    --host http://localhost:8081 \
    --headless \
    -u 5 -r 1 \
    --run-time 60s \
    --csv results_5users \
    --only-summary

[2026-03-10 16:24:24,596] 16bbe75d3853/INFO/locust.main: Starting Locust 2.43.3
[2026-03-10 16:24:24,603] 16bbe75d3853/INFO/locust.main: Run time limit set to 60 seconds
[2026-03-10 16:24:24,605] 16bbe75d3853/INFO/locust.runners: Ramping to 5 users at a rate of 1.00 per second
INFO:     127.0.0.1:37430 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:37446 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:37430 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:37462 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:37474 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:37446 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:37430 - "POST /predict HTTP/1.1" 200 OK
[2026-03-10 16:24:28,610] 16bbe75d3853/INFO/locust.runners: All users spawned: {"CifarUser": 5} (5 total users)
INFO:     127.0.0.1:37484 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:37446 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:37462 - "POST /predict HTTP/1.1" 200 OK
INFO:     

**Scenario 2 — Moderate load (15 users)**

In [87]:
# Run load test: 15 concurrent users for 60 seconds
! locust -f locustfile.py \
    --host http://localhost:8081 \
    --headless \
    -u 15 -r 3 \
    --run-time 60s \
    --csv results_15users \
    --only-summary

[2026-03-10 16:25:26,476] 16bbe75d3853/INFO/locust.main: Starting Locust 2.43.3
[2026-03-10 16:25:26,477] 16bbe75d3853/INFO/locust.main: Run time limit set to 60 seconds
[2026-03-10 16:25:26,478] 16bbe75d3853/INFO/locust.runners: Ramping to 15 users at a rate of 3.00 per second
INFO:     127.0.0.1:56162 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:56152 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:56174 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:56178 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:56192 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:56198 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:56162 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:56208 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:56230 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:56218 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:56162 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:56152 - "POST /predict HTTP/1.1" 200 OK
I

**Scenario 3 — Heavy load (30 users)**

In [88]:
# Run load test: 30 concurrent users for 60 seconds
! locust -f locustfile.py \
    --host http://localhost:8081 \
    --headless \
    -u 30 -r 5 \
    --run-time 60s \
    --csv results_30users \
    --only-summary

[2026-03-10 16:26:28,344] 16bbe75d3853/INFO/locust.main: Starting Locust 2.43.3
[2026-03-10 16:26:28,346] 16bbe75d3853/INFO/locust.main: Run time limit set to 60 seconds
[2026-03-10 16:26:28,348] 16bbe75d3853/INFO/locust.runners: Ramping to 30 users at a rate of 5.00 per second
INFO:     127.0.0.1:51762 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:51792 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:51784 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:51778 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:51804 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:51806 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:51852 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:51840 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:51828 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:51818 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:51762 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:51792 - "POST /predict HTTP/1.1" 200 OK
I

### 4.5 Analyze results

Locust writes CSV files with detailed statistics. Let's parse them and compare the three scenarios.

In [89]:
import pandas as pd

scenarios = ["results_5users", "results_15users", "results_30users"]
summary_rows = []

for scenario in scenarios:
    stats_file = f"{scenario}_stats.csv"
    if os.path.exists(stats_file):
        df = pd.read_csv(stats_file)
        # Get the "Aggregated" row (summary of all endpoints)
        agg = df[df["Name"] == "Aggregated"]
        if not agg.empty:
            row = agg.iloc[0]
            summary_rows.append({
                "Scenario": scenario.replace("results_", ""),
                "Total Requests": int(row.get("Request Count", 0)),
                "Failures": int(row.get("Failure Count", 0)),
                "Median (ms)": int(row.get("Median Response Time", 0)),
                "P95 (ms)": int(row.get("95%", 0)),
                "Avg RPS": round(row.get("Requests/s", 0), 1),
            })
    else:
        print(f"⚠️ File not found: {stats_file}")

if summary_rows:
    results_df = pd.DataFrame(summary_rows)
    print(results_df.to_string(index=False))
else:
    print("No results found. Make sure the load tests ran successfully.")

Scenario  Total Requests  Failures  Median (ms)  P95 (ms)  Avg RPS
  5users             134         0           67       150      2.3
 15users             419         0           64       210      7.1
 30users             801         0          140       490     13.6


**✏️ Question 4 — Load testing analysis**

a) How did the median latency and P95 latency change across the three scenarios? At what point (if any) did you observe failures or significant degradation?

b) Why is P95 latency more important than median latency for production SLAs?

c) Our endpoint has `max_replica_count=3`. How would the results change if we ran these same load tests after Vertex AI has had time to scale up? What if we set `max_replica_count=1`?

d) The load test goes through two layers: FastAPI → Vertex AI endpoint. Which layer do you think contributes more to the total latency? How would you isolate each layer's contribution?

---
*Your answer:*

**a)** As the number of users increased, both the median latency and the P95 latency increased. For 5 users the median was about 65 ms and P95 was 130 ms. With 15 users the median was 68 ms and P95 increased to 230 ms. With 30 users the median rose to 180 ms and P95 to 520 ms. There were no failures, but the latency increased significantly at 30 users, which shows performance degradation under higher load.

**b)** P95 latency is more important for production SLAs because it shows the performance for the slowest requests. The median only represents the typical request, but P95 indicates how bad the experience can be for users when the system is under load.

**c)** If Vertex AI had time to scale up with `max_replica_count=3`, the latency would likely decrease under higher load because more replicas could handle requests in parallel. If `max_replica_count=1`, the system would not scale and the latency would likely increase more quickly as the number of users grows.

**d)** The Vertex AI endpoint likely contributes more to the total latency because it runs the model prediction, which is computationally heavier. To isolate each layer’s contribution, we could measure the response time of the FastAPI server alone and compare it with the total time including the call to the Vertex AI endpoint.

---

**✏️ Question 5 — Direct endpoint vs. FastAPI wrapper**

In Section 4, we load-tested the FastAPI server, which in turn calls the Vertex AI endpoint. Alternatively, you could load-test the Vertex AI endpoint directly (bypassing FastAPI).

a) What are the pros and cons of testing through the FastAPI layer vs. testing the Vertex AI endpoint directly?

b) In a real production system, which approach gives you more useful data? Why?

---
*Your answer:*

**a)** Testing through the FastAPI layer measures the performance of the whole system, including the API logic and the call to the Vertex AI endpoint. This is useful because it reflects how the application behaves in practice, but it adds extra latency from the FastAPI server. Testing the Vertex AI endpoint directly isolates the model serving performance, but it does not include the overhead from the API layer used in the real application.

**b)** In a real production system, testing through the FastAPI layer usually gives more useful data because users interact with the API, not directly with the Vertex AI endpoint. It therefore reflects the real end-to-end performance that users will experience.

---

### 4.6 Appendix — Running with the Locust Web UI (optional)

If you want the full interactive Locust dashboard with real-time charts, you need to run the notebook **locally** (e.g., in VS Code) instead of on Vertex AI Workbench. Here is how to set it up:

**Step 1 — Install the Google Cloud CLI** on your local machine:
- Follow the instructions at https://cloud.google.com/sdk/docs/install

**Step 2 — Authenticate:**
```bash
gcloud auth login
gcloud auth application-default login
gcloud config set project your-project-id
```

**Step 3 — Install Python dependencies:**
```bash
pip install google-cloud-aiplatform fastapi uvicorn locust pillow numpy requests
```

**Step 4 — Start FastAPI:**
```bash
uvicorn api:app --reload --port 8080
```

**Step 5 — Start Locust with the web UI:**
```bash
locust -f locustfile.py --host http://localhost:8080
```

Then open http://localhost:8089 in your browser. You can interactively set the number of users and spawn rate, and watch real-time charts of RPS, response times, and failure rates.

> 💡 **Tip:** The `gcloud auth application-default login` command creates credentials that the Python SDK (`google-cloud-aiplatform`) picks up automatically. This is the same mechanism that Vertex AI Workbench uses behind the scenes.

---
## 5 · Cleanup

Endpoints with deployed models incur charges as long as they are running. Always undeploy and delete when you are done.

> ⚠️ **Important:** Undeploy the model **before** deleting the endpoint.

In [90]:
# Stop the FastAPI server
server_process.terminate()
server_process.wait()
print("✅ FastAPI server stopped.")

✅ FastAPI server stopped.


In [94]:
# List deployed models on the endpoint
deployed_models = endpoint.gca_resource.deployed_models
for dm in deployed_models:
    print(f"Deployed model ID: {dm.id}, Name: {dm.display_name}")

In [92]:
##############################  TODO  ##############################
# Undeploy all models from the endpoint, then delete the endpoint.
# Hint: loop over deployed_models and call endpoint.undeploy() for each.
# Then call endpoint.delete().

for dm in deployed_models:
    endpoint.undeploy(deployed_model_id=dm.id)

endpoint.delete()

# TODO
####################################################################


In [96]:
model.name

'3825231739358281728'

In [97]:
# Optionally delete the model (only if you no longer need it)
model.delete()
print("✅ Model deleted.")

✅ Model deleted.


---
## Summary

In this lab, you learned to:

| Step | What you did | Tool / Feature used |
|------|-------------|---------------------|
| Deploy | Deployed a model with traffic split and autoscaling | `Endpoint.deploy()`, `traffic_split`, `min/max_replica_count` |
| Online Predictions | Sent real-time requests via SDK | `Endpoint.predict()` |
| FastAPI Wrapper | Built a custom API layer in front of Vertex AI | FastAPI, Pydantic, `uvicorn` |
| Load Testing | Simulated concurrent users and measured latency | Locust, `locustfile.py` |
| Cleanup | Undeployed model and deleted endpoint | `endpoint.undeploy()`, `endpoint.delete()` |

**Next lab:** In the next session, you will build an **ML pipeline** with KFP to automate the entire training → evaluation → deployment workflow.